In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Combine Files into a Single DataFrame

This section will:
1. List all files in the specified folder.
2. Read each CSV file into a pandas DataFrame.
3. Concatenate all DataFrames into a single one.
4. Sort the combined DataFrame by the 'Time' column.

In [ ]:
import os

# Define the folder path where your files are located
folder_path = '/content/drive/Shareddrives/Data Team/ABMF 2026 Historical Data'

# List all files in the folder
file_names = [f for f in os.listdir(folder_path)]

# Initialize an empty list to store individual DataFrames
all_dfs = []

# Loop through each file, read it, and append to the list
for file in file_names:
    file_path = os.path.join(folder_path, file)
    try:
        df = pd.read_csv(file_path)
        all_dfs.append(df)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

# Concatenate all DataFrames into a single one
if all_dfs:
    combined_df = pd.concat(all_dfs, ignore_index=True)

    # Sort the combined DataFrame by the 'Time' column
    # Assuming 'Time' column exists and is parsable. Adjust if column name is different.
    if 'Time' in combined_df.columns:
        combined_df['Time'] = pd.to_datetime(combined_df['Time'], errors='coerce')
        combined_df = combined_df.sort_values(by='Time').reset_index(drop=True)
        print("Successfully combined and sorted the files.")
        display(combined_df.head())
    else:
        print("The 'Time' column was not found for sorting.")
        display(combined_df.head())
else:
    print(f"No CSV files found in the folder: {folder_path}")

In [ ]:
combined_df.shape

### Extract Date and Count Rows per Day

This section will:
1. Extract the date component from the `Time` column.
2. Count the number of rows for each unique date.

In [ ]:
# Extract the date from the 'Time' column
combined_df['Date'] = combined_df['Time'].dt.date

# Count the number of rows for each unique date
daily_row_counts = combined_df['Date'].value_counts().sort_index()

print("Number of rows per day:")
display(daily_row_counts)

In [ ]:
combined_df.columns.tolist()

In [ ]:
combined_df[[ 'Pa_kW', 'Pb_kW', 'Pc_kW', 'P_kW']]

In [ ]:
combined_df['Pa_kWh'] = combined_df['Pa_kW'] * 1 / 60

In [ ]:
combined_df['Pb_kWh'] = combined_df['Pb_kW'] * 1 / 60

In [ ]:
combined_df['Pc_kWh'] = combined_df['Pc_kW'] * 1 / 60

In [ ]:
combined_df['P_kWh'] = combined_df['P_kW'] * 1 / 60

### Categorize Power Source (Grid/Generator/Off)

This section will create a new column named `Power_Source` in `combined_df` based on the following logic:
- If `I001_A`, `I002_A`, or `I003_A` is not zero, the `Power_Source` will be 'Grid'.
- Else if `I004_A`, `I005_A`, or `I006_A` is not zero, the `Power_Source` will be 'Generator'.
- Otherwise, the `Power_Source` will be 'Off'.

In [ ]:
# Define the conditions
conditions = [
    (combined_df['I001_A'] != 0) | (combined_df['I004_A'] != 0) | (combined_df['I007_A'] != 0),
    (combined_df['I002_A'] != 0) | (combined_df['I005_A'] != 0) | (combined_df['I008_A'] != 0)
]

# Define the choices corresponding to the conditions
choices = ['Grid', 'Generator']

# Apply the conditions using np.select to create the new 'Power_Source' column
combined_df['Power_Source'] = np.select(conditions, choices, default='Off')

In [ ]:
# Display the first few rows with the new column and its value counts
print("Combined DataFrame with 'Power_Source' column:")
display(combined_df[['I001_A', 'I002_A', 'I003_A', 'I004_A', 'I005_A', 'I006_A', 'I007_A', 'I008_A', 'I009_A', 'Power_Source']].head())

print("\nValue counts for 'Power_Source' column:")
display(combined_df['Power_Source'].value_counts())

### Total P_kWh per Month by Power Source

This section will:
1. Extract the month from the `Time` column.
2. Group the data by month and `Power_Source`.
3. Calculate the sum of `P_kWh` for each group.

In [ ]:
# Extract the month from the 'Time' column
combined_df['Month'] = combined_df['Time'].dt.to_period('M')

# Group by Month and Power_Source and sum 'P_kWh'
monthly_kwh_by_source = combined_df.groupby(['Month', 'Power_Source'])['P_kWh'].sum().unstack(fill_value=0)

print("Total P_kWh for each month by Power Source:")
display(monthly_kwh_by_source)

### Filtered Monthly P_kWh and Totals (Jan-Apr 2026)

This section will:
1. Filter the `monthly_kwh_by_source` DataFrame for January, February, March, and April 2026.
2. Calculate the total `P_kWh` for each `Power_Source` during these months.
3. Calculate the total `P_kWh` for each of these months across all `Power_Sources`.

In [ ]:
# Define the months for which we want to calculate the total
selected_months = pd.PeriodIndex(['2026-01', '2026-02', '2026-03', '2026-04'], freq='M')

# Filter the monthly_kwh_by_source DataFrame for the selected months
filtered_monthly_kwh = monthly_kwh_by_source.loc[monthly_kwh_by_source.index.isin(selected_months)]

# Calculate the 'Total' column for P_kWh for each month
filtered_monthly_kwh['Total'] = filtered_monthly_kwh.sum(axis=1)

# Calculate the 'Grand Total' row for P_kWh across selected months
monthly_totals = filtered_monthly_kwh.sum().to_frame().T
monthly_totals.index = ['Grand Total']

# Combine the filtered monthly data with the grand total row
final_monthly_summary = pd.concat([filtered_monthly_kwh, monthly_totals])

print("Total P_kWh for Jan, Feb, March, and April 2026 by Power Source, with totals:")
display(final_monthly_summary)

In [ ]:
combined_df.columns

In [ ]:
combined_df[['I001_A', 'I002_A', 'I003_A', 'I004_A', 'I005_A', 'I006_A', 'I007_A', 'I008_A', 'I009_A']].sample(10)

In [ ]:
combined_df.head(100)

02/07

### Daily P_kWh by Power Source and Month Visualization

This section will generate stacked bar charts for each month, displaying the daily `P_kWh` consumption categorized by 'Grid' and 'Generator' power sources.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Ensure the 'Date' column is present and correctly formatted
combined_df['Date'] = combined_df['Time'].dt.date

# Group by Date and Power_Source to get daily P_kWh for each source
daily_kwh_by_source = combined_df.groupby(['Date', 'Power_Source'])['P_kWh'].sum().unstack(fill_value=0)

# Convert the 'Date' index to datetime for easier plotting
daily_kwh_by_source.index = pd.to_datetime(daily_kwh_by_source.index)

# Extract month for grouping plots
daily_kwh_by_source['Month'] = daily_kwh_by_source.index.to_period('M')

# Filter for months January to April 2026
selected_periods = [pd.Period('2026-01', 'M'), pd.Period('2026-02', 'M'),
                    pd.Period('2026-03', 'M'), pd.Period('2026-04', 'M')]
unique_months = np.sort([p for p in daily_kwh_by_source['Month'].unique() if p in selected_periods])

# Define custom colors
custom_colors = {
    'Grid': '#0094B8',
    'Generator': '#4E46BE'
}

for month in unique_months:
    month_data = daily_kwh_by_source[daily_kwh_by_source['Month'] == month].drop(columns=['Month'])

    if not month_data.empty:
        # Ensure 'Grid' and 'Generator' columns exist, fill with 0 if not present for a month
        if 'Grid' not in month_data.columns: month_data['Grid'] = 0
        if 'Generator' not in month_data.columns: month_data['Generator'] = 0
        # Drop 'Off' column if it exists and is all zeros, or if it's not needed for the chart
        if 'Off' in month_data.columns and month_data['Off'].sum() == 0: # Check if 'Off' column is all zeros
             month_data = month_data.drop(columns=['Off'])

        # Create stacked bar chart
        fig, ax = plt.subplots(figsize=(15, 7))
        # Use only 'Grid' and 'Generator' columns for plotting
        plot_data = month_data[['Grid', 'Generator']]
        # Use custom colors
        plot_data.plot(kind='bar', stacked=True, ax=ax, color=[custom_colors[col] for col in plot_data.columns])

        ax.set_title(f'Daily P_kWh Consumption by Power Source for {month}')
        ax.set_xlabel('Date')
        ax.set_ylabel('Total P_kWh')
        ax.tick_params(axis='x', rotation=45)

        # Format x-axis labels to 'Weekday Mon Day' (e.g., 'Mon Jan 01')
        ax.set_xticklabels([d.strftime('%a %b %d') for d in month_data.index])

        # Add totals and individual source values on bars
        for i, (idx, row) in enumerate(plot_data.iterrows()):
            grid_val = row['Grid']
            gen_val = row['Generator']
            total_val = grid_val + gen_val

            # Annotate Grid value (bottom of stack)
            if grid_val > 0: # Only annotate if there's a value to show
                ax.text(i, grid_val / 2, f'{grid_val:.0f}', ha='center', va='center', color='white', fontsize=8)

            # Annotate Generator value (top of stack)
            if gen_val > 0: # Only annotate if there's a value to show
                ax.text(i, grid_val + gen_val / 2, f'{gen_val:.0f}', ha='center', va='center', color='white', fontsize=8)

            # Annotate total value (above the entire stack)
            if total_val > 0: # Only annotate if there's a value to show
                ax.text(i, total_val + 5, f'{total_val:.0f}', ha='center', va='bottom', color='black', fontsize=9)

        ax.legend(title='Power Source')
        plt.tight_layout()
        plt.show()
    else:
        print(f"No data for month: {month}")

### Calculate Specific Fuel Consumption (SPC) and Diesel Consumed for Generator

This section will:
1. Calculate a `spc` (specific fuel consumption) value for rows where `Power_Source` is 'Generator', using the formula: `3.3323 / P_kW + 0.1160`.
2. Calculate `Diesel_Consumed` by multiplying `spc` by `P_kWh` for 'Generator' power source.

In [ ]:
# Initialize 'spc' and 'Diesel_Consumed' columns with 0 or NaN
combined_df['spc'] = np.nan
combined_df['Diesel_Consumed'] = np.nan

# Identify rows where Power_Source is 'Generator'
gen_condition = (combined_df['Power_Source'] == 'Generator')

# Calculate 'spc' only for 'Generator' rows, handling division by zero
# If P_kW is 0, spc will be np.nan to avoid ZeroDivisionError
combined_df.loc[gen_condition, 'spc'] = np.where(
    combined_df.loc[gen_condition, 'P_kW'] != 0,
    3.3323 / combined_df.loc[gen_condition, 'P_kW'] + 0.1160,
    np.nan
)

# Calculate 'Diesel_Consumed' for 'Generator' rows where 'spc' is not NaN
combined_df.loc[gen_condition, 'Diesel_Consumed'] = combined_df.loc[gen_condition, 'spc'] * combined_df.loc[gen_condition, 'P_kWh']

print("First few rows with new 'spc' and 'Diesel_Consumed' columns:")
display(combined_df[['Power_Source', 'P_kW', 'P_kWh', 'spc', 'Diesel_Consumed']].head(10))

print("\nSummary statistics for 'spc' and 'Diesel_Consumed' for Generator power source:")
display(combined_df[gen_condition][['spc', 'Diesel_Consumed']].describe())

### Total Diesel Consumed for Each Month (Jan-Apr 2026)

This section calculates the total `Diesel_Consumed` for each month, specifically for the 'Generator' power source, for January to April 2026.

In [ ]:
# Define the months for which we want to calculate total diesel consumption
selected_months_str = ['2026-01', '2026-02', '2026-03', '2026-04']
selected_months_period = pd.PeriodIndex(selected_months_str, freq='M')

# Ensure 'Month' column is available
combined_df['Month'] = combined_df['Time'].dt.to_period('M')

# Filter combined_df for 'Generator' power source and selected months
diesel_data_filtered = combined_df[
    (combined_df['Power_Source'] == 'Generator') &
    (combined_df['Month'].isin(selected_months_period))
].copy() # Use .copy() to avoid SettingWithCopyWarning

# Group by Month and sum 'Diesel_Consumed'
total_diesel_consumed_monthly = diesel_data_filtered.groupby('Month')['Diesel_Consumed'].sum()

print("Total Diesel Consumed for each month (January - April 2026):")
display(total_diesel_consumed_monthly)

### Calculate Generator Running Hours per Month (Jan-Apr 2026)

This section will calculate the total running hours for the 'Generator' for each month from January to April 2026. Assuming each data point represents one minute of operation, running hours are calculated by counting the number of 'Generator' entries and dividing by 60.

In [ ]:
# Filter combined_df for 'Generator' power source and selected months (already defined in previous cell)
# diesel_data_filtered contains the relevant data for 'Generator' in Jan-Apr 2026

# Group by Month and count the number of entries for 'Generator'
# Each entry represents 1 minute of operation
generator_minutes_monthly = diesel_data_filtered.groupby('Month').size()

# Convert minutes to hours (1 hour = 60 minutes)
generator_running_hours_monthly = generator_minutes_monthly / 60

print("Generator Running Hours for each month (January - April 2026):")
display(generator_running_hours_monthly)